# Feature Engineering: Bronze → Silver → Gold Pipeline
**Input:** data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv (Bronze)  
**Output:** data/silver/telco_silver.parquet · data/gold/telco_gold.parquet  
**Goal:** Transform raw data into ML-ready features using industry-standard pipeline

In [1]:
import pandas as pd
import numpy as np

# Load the Bronze layer (raw CSV — never modify this directly)
df_bronze = pd.read_csv('C:/Users/DELL/Documents/telecom-churn-analytics/data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')

print(f"✓ Bronze layer loaded")
print(f"  Shape  : {df_bronze.shape}")
print(f"  Columns: {list(df_bronze.columns)}")

✓ Bronze layer loaded
  Shape  : (7043, 21)
  Columns: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


## Bronze → Silver
Cleaning and standardising the raw data:
1. Fix TotalCharges (text → number)
2. Convert Churn to binary (Yes/No → 1/0)
3. Convert all Yes/No columns to 1/0
4. Simplify "No internet service" and "No phone service" to just "No"

In [2]:
# Always copy — never modify the bronze dataframe directly
df_silver = df_bronze.copy()

# Fix 1: TotalCharges text → number (same fix as Day 1)
df_silver['TotalCharges'] = pd.to_numeric(
    df_silver['TotalCharges'], errors='coerce'
).fillna(0)

print("Fix 1 ✓ TotalCharges converted to float")
print(f"  dtype now: {df_silver['TotalCharges'].dtype}")

Fix 1 ✓ TotalCharges converted to float
  dtype now: float64


In [3]:
# Fix 2: Churn column → binary (this becomes our TARGET variable for ML)
# Yes = customer left = 1
# No  = customer stayed = 0
df_silver['Churn'] = (df_silver['Churn'] == 'Yes').astype(int)

print("Fix 2 ✓ Churn converted to binary")
print(df_silver['Churn'].value_counts())

Fix 2 ✓ Churn converted to binary
Churn
0    5174
1    1869
Name: count, dtype: int64


In [4]:
# Fix 3: Convert binary Yes/No columns → 0/1
# These 6 columns all contain only 'Yes' or 'No'
binary_cols = [
    'Partner', 
    'Dependents', 
    'PhoneService', 
    'PaperlessBilling'
]

for col in binary_cols:
    df_silver[col] = (df_silver[col] == 'Yes').astype(int)
    print(f"  ✓ {col}: Yes→1, No→0")

print(f"\nFix 3 ✓ All binary columns converted")

  ✓ Partner: Yes→1, No→0
  ✓ Dependents: Yes→1, No→0
  ✓ PhoneService: Yes→1, No→0
  ✓ PaperlessBilling: Yes→1, No→0

Fix 3 ✓ All binary columns converted


In [5]:
# Fix 4: Replace "No internet service" and "No phone service" with "No"
# These aren't meaningful distinctions for our analysis — both mean the 
# customer doesn't have the feature
internet_cols = [
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies'
]

for col in internet_cols:
    df_silver[col] = df_silver[col].replace(
        'No internet service', 'No'
    )
    print(f"  ✓ {col}: 'No internet service' → 'No'")

# Same for MultipleLines
df_silver['MultipleLines'] = df_silver['MultipleLines'].replace(
    'No phone service', 'No'
)
print(f"  ✓ MultipleLines: 'No phone service' → 'No'")

print("\nFix 4 ✓ All service columns simplified")

  ✓ OnlineSecurity: 'No internet service' → 'No'
  ✓ OnlineBackup: 'No internet service' → 'No'
  ✓ DeviceProtection: 'No internet service' → 'No'
  ✓ TechSupport: 'No internet service' → 'No'
  ✓ StreamingTV: 'No internet service' → 'No'
  ✓ StreamingMovies: 'No internet service' → 'No'
  ✓ MultipleLines: 'No phone service' → 'No'

Fix 4 ✓ All service columns simplified


In [6]:
print("=" * 45)
print("SILVER LAYER — Quality Check")
print("=" * 45)
print(f"Shape          : {df_silver.shape}")
print(f"Null values    : {df_silver.isnull().sum().sum()}")
print(f"Duplicates     : {df_silver.duplicated().sum()}")
print(f"\nChurn rate     : {df_silver['Churn'].mean()*100:.1f}%")
print(f"TotalCharges   : {df_silver['TotalCharges'].dtype}")
print(f"\nSample of Partner column: {df_silver['Partner'].unique()}")
print(f"Sample of Churn column  : {df_silver['Churn'].unique()}")

SILVER LAYER — Quality Check
Shape          : (7043, 21)
Null values    : 0
Duplicates     : 0

Churn rate     : 26.5%
TotalCharges   : float64

Sample of Partner column: [1 0]
Sample of Churn column  : [0 1]


In [7]:
# Save Silver layer as Parquet
silver_path = 'C:/Users/DELL/Documents/telecom-churn-analytics/data/silver/telco_silver.parquet'
df_silver.to_parquet(silver_path, index=False)

# Verify it was saved and check size
import os
file_size = os.path.getsize(silver_path) / 1024
print(f"✓ Silver layer saved to: {silver_path}")
print(f"  File size: {file_size:.1f} KB")

# Compare to original CSV
csv_size = os.path.getsize(
    'C:/Users/DELL/Documents/telecom-churn-analytics/data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv'
) / 1024
print(f"  Original CSV size: {csv_size:.1f} KB")
print(f"  Parquet is {csv_size/file_size:.1f}x smaller than CSV")

✓ Silver layer saved to: C:/Users/DELL/Documents/telecom-churn-analytics/data/silver/telco_silver.parquet
  File size: 187.1 KB
  Original CSV size: 954.6 KB
  Parquet is 5.1x smaller than CSV


## Silver → Gold
Engineering 3 new features that are more informative than raw columns:

1. **tenure_group** — buckets tenure into business-meaningful bands
2. **avg_monthly_spend** — spend efficiency (TotalCharges ÷ tenure)
3. **service_count** — how many services the customer subscribes to

These features are designed based on business intuition from the 
Day 1 EDA findings — they're not random transformations.

In [8]:
# Always build Gold from Silver — never from Bronze
df_gold = df_silver.copy()

# New Feature 1: tenure_group
# WHY: Raw tenure (0-72 months) is hard to interpret in business terms.
# Bucketing into bands makes the "churn happens early" insight actionable.
df_gold['tenure_group'] = pd.cut(
    df_gold['tenure'],
    bins=[0, 12, 24, 48, 72],
    labels=['0-1 Year', '1-2 Years', '2-4 Years', '4-6 Years'],
    include_lowest=True
)

print("New Feature 1 ✓ tenure_group")
print(df_gold['tenure_group'].value_counts().sort_index())

New Feature 1 ✓ tenure_group
tenure_group
0-1 Year     2186
1-2 Years    1024
2-4 Years    1594
4-6 Years    2239
Name: count, dtype: int64


In [9]:
# New Feature 2: avg_monthly_spend
# WHY: TotalCharges alone is misleading — a customer with tenure=60 
# SHOULD have high TotalCharges. What matters is how much they pay 
# per month on average vs their current MonthlyCharges.
# We add 1 to tenure to avoid division by zero for new customers.

df_gold['avg_monthly_spend'] = (
    df_gold['TotalCharges'] / (df_gold['tenure'] + 1)
).round(2)

print("New Feature 2 ✓ avg_monthly_spend")
print(f"  Min: ${df_gold['avg_monthly_spend'].min():.2f}")
print(f"  Max: ${df_gold['avg_monthly_spend'].max():.2f}")
print(f"  Mean: ${df_gold['avg_monthly_spend'].mean():.2f}")

New Feature 2 ✓ avg_monthly_spend
  Min: $0.00
  Max: $118.97
  Mean: $58.99


In [10]:
# New Feature 3: service_count
# WHY: Customers subscribed to MORE services are harder to leave 
# (more value, more switching cost). This "bundle effect" was 
# hinted at in Day 1 — let's quantify it as a feature.

# Count services where value is 'Yes' across 7 service columns
service_cols = [
    'PhoneService', 'MultipleLines', 'OnlineSecurity',
    'OnlineBackup', 'DeviceProtection', 'TechSupport',
    'StreamingTV', 'StreamingMovies'
]

df_gold['service_count'] = (
    df_gold[service_cols] == 'Yes'
).sum(axis=1)

# axis=1 means "count across columns for each row" (not down the column)
print("New Feature 3 ✓ service_count")
print(df_gold['service_count'].value_counts().sort_index())

New Feature 3 ✓ service_count
service_count
0    1667
1    1158
2     957
3     978
4     933
5     722
6     420
7     208
Name: count, dtype: int64


## One-Hot Encoding
Some columns have more than 2 categories:
- **Contract**: Month-to-month, One year, Two year
- **InternetService**: DSL, Fiber optic, No
- **PaymentMethod**: 4 different methods

We can't convert these to 0/1 directly (that would imply 
"Month-to-month=0 < One year=1 < Two year=2" which is wrong —
they're just different categories with no order).

One-hot encoding creates a NEW column for each category value.
For example, Contract becomes:
- Contract_One year    → 1 if customer has one year, else 0
- Contract_Two year    → 1 if customer has two year, else 0
(Month-to-month is the reference — if both are 0, it's month-to-month)

In [11]:
# One-hot encode the 3 multi-category columns
# drop_first=True removes one category per column to avoid redundancy
# (if Contract_One year=0 AND Contract_Two year=0, we KNOW it's Month-to-month)

df_gold = pd.get_dummies(
    df_gold,
    columns=['Contract', 'PaymentMethod', 'InternetService'],
    drop_first=True
)

print("✓ One-hot encoding applied")
print(f"  New shape: {df_gold.shape}")
print(f"\nNew columns created:")
new_cols = [c for c in df_gold.columns 
            if 'Contract' in c or 'Payment' in c or 'Internet' in c]
for col in new_cols:
    print(f"  → {col}")

✓ One-hot encoding applied
  New shape: (7043, 28)

New columns created:
  → Contract_One year
  → Contract_Two year
  → PaymentMethod_Credit card (automatic)
  → PaymentMethod_Electronic check
  → PaymentMethod_Mailed check
  → InternetService_Fiber optic
  → InternetService_No


In [12]:
# Save Gold layer as Parquet
gold_path = 'C:/Users/DELL/Documents/telecom-churn-analytics/data/gold/telco_gold.parquet'
df_gold.to_parquet(gold_path, index=False)

gold_size = os.path.getsize(gold_path) / 1024
print(f"✓ Gold layer saved to: {gold_path}")
print(f"  Rows   : {df_gold.shape[0]:,}")
print(f"  Columns: {df_gold.shape[1]}")
print(f"  Size   : {gold_size:.1f} KB")
print(f"\nPipeline complete!")
print(f"  Bronze: 21 cols, text categories, CSV")
print(f"  Silver: 21 cols, cleaned, Parquet")
print(f"  Gold  : {df_gold.shape[1]} cols, ML-ready, Parquet")

✓ Gold layer saved to: C:/Users/DELL/Documents/telecom-churn-analytics/data/gold/telco_gold.parquet
  Rows   : 7,043
  Columns: 28
  Size   : 229.8 KB

Pipeline complete!
  Bronze: 21 cols, text categories, CSV
  Silver: 21 cols, cleaned, Parquet
  Gold  : 28 cols, ML-ready, Parquet


In [13]:
print("=" * 50)
print("FEATURE ENGINEERING — Complete Summary")
print("=" * 50)
print(f"\nBronze → Silver transformations:")
print(f"  ✓ TotalCharges: text → float")
print(f"  ✓ Churn: Yes/No → 1/0")
print(f"  ✓ 4 binary columns: Yes/No → 1/0")
print(f"  ✓ 'No internet service' simplified to 'No'")
print(f"\nSilver → Gold transformations:")
print(f"  ✓ tenure_group: new categorical feature")
print(f"  ✓ avg_monthly_spend: new numeric feature")
print(f"  ✓ service_count: new count feature")
print(f"  ✓ One-hot encoding: Contract, PaymentMethod, InternetService")
print(f"\nFiles saved:")
print(f"  ✓ data/silver/telco_silver.parquet")
print(f"  ✓ data/gold/telco_gold.parquet")
print("=" * 50)

FEATURE ENGINEERING — Complete Summary

Bronze → Silver transformations:
  ✓ TotalCharges: text → float
  ✓ Churn: Yes/No → 1/0
  ✓ 4 binary columns: Yes/No → 1/0
  ✓ 'No internet service' simplified to 'No'

Silver → Gold transformations:
  ✓ tenure_group: new categorical feature
  ✓ avg_monthly_spend: new numeric feature
  ✓ service_count: new count feature
  ✓ One-hot encoding: Contract, PaymentMethod, InternetService

Files saved:
  ✓ data/silver/telco_silver.parquet
  ✓ data/gold/telco_gold.parquet
